In [1]:
pip install -Uqq langchain-groq

In [2]:
pip install -Uqq groq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.3 requires groq<1.0.0,>=0.30.0, but you have groq 1.7.0 which is incompatible.


In [3]:
pip install -Uqq langchain.tools

In [12]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import langchain_groq
import os
from google.colab import userdata
import requests
from langchain.tools import tool
from groq import Groq
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

In [13]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [14]:
client = Groq()
for model in client.models.list().data:
  print(model.id)

meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
whisper-large-v3
canopylabs/orpheus-v1-english
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
allam-2-7b


In [15]:
@tool
def get_weather(latitude: float, longitude: float) -> dict:
    """Get current weather for a location using latitude and longitude."""

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m",
        "temperature_unit": "celsius"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()["current"]

In [16]:
class weather_report(BaseModel):
  city: str
  temperature_2m: float = Field(
      description="Current air temperature in degrees Celsius"
  )
  relative_humidity_2m: float = Field(
      description="Current relative humidity as a percentage"
  )

In [17]:
agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools = [get_weather],
    system_prompt="You are a helpful assistant",
)

In [25]:
llm = init_chat_model("groq:openai/gpt-oss-20b")
llm_structured = llm.with_structured_output(weather_report)
def reasoning():
  request = input("Enter the city: ")
  if request == 'stop':
    return 'stop'
  result = agent.invoke({
      "messages":[{
        "role":"user",
        "content":f"give me the current weather in {request}"
      }]
  })
  unstructured_result = result['messages'][-1].content
  structured_result = llm_structured.invoke(f"Extract the weather report from this text. Do not reply to it {unstructured_result}")
  return structured_result

In [27]:
while True:
  answer = reasoning()
  if isinstance(answer, str):
    break
  else:
    print(answer)

Enter the city: Cairo
city='Cairo' temperature_2m=19.2 relative_humidity_2m=91.0
Enter the city: London
city='London' temperature_2m=10.9 relative_humidity_2m=67.0
Enter the city: stop
